In [1]:
import mysql.connector
import pandas as pd
from getpass import getpass

# Enter your MySQL password securely
db_password = getpass("Enter MySQL password: ")

# Connect to MySQL
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password=db_password,
    database="ecommerce_intelligence"
)

print("MySQL connection successful!")

MySQL connection successful!


In [2]:
%pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import mysql.connector

print("MySQL connector working!")

MySQL connector working!


In [4]:
import mysql.connector

print("MySQL connector working!")

MySQL connector working!


In [5]:
import mysql.connector
import pandas as pd
from getpass import getpass

db_password = getpass("Enter MySQL password: ")

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password=db_password,
    database="ecommerce_intelligence"
)

print("MySQL connection successful!")

MySQL connection successful!


In [6]:
query = """
SELECT
    review_id,
    product_id,
    review_date,
    rating,
    review_text
FROM reviews
ORDER BY review_id;
"""

cursor = connection.cursor()
cursor.execute(query)

rows = cursor.fetchall()

columns = [
    "review_id",
    "product_id",
    "review_date",
    "rating",
    "review_text"
]

reviews_df = pd.DataFrame(rows, columns=columns)

cursor.close()
connection.close()

print("Total reviews fetched:", len(reviews_df))
reviews_df.head()

Total reviews fetched: 250


,review_id,product_id,review_date,rating,review_text
0,R0001,P006,2025-06-27,3,Average quality for the price.
1,R0002,P017,2025-05-09,5,Excellent quality and works perfectly.
2,R0003,P007,2025-05-21,4,Exactly as described. Very satisfied.
3,R0004,P005,2025-10-01,4,Exactly as described. Very satisfied.
4,R0005,P003,2025-10-25,4,Very happy with this purchase.


In [7]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

print("VADER sentiment analyzer ready!")

VADER sentiment analyzer ready!


In [8]:
sample_review = reviews_df.loc[0, "review_text"]

sentiment_score = analyzer.polarity_scores(sample_review)

print("Review:", sample_review)
print("Sentiment scores:", sentiment_score)

Review: Average quality for the price.
Sentiment scores: {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}


In [9]:
def get_sentiment(text):
    score = analyzer.polarity_scores(str(text))["compound"]

    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"


reviews_df["sentiment_score"] = reviews_df["review_text"].apply(
    lambda text: analyzer.polarity_scores(str(text))["compound"]
)

reviews_df["sentiment"] = reviews_df["review_text"].apply(get_sentiment)

reviews_df[["review_text", "rating", "sentiment_score", "sentiment"]].head(10)

,review_text,rating,sentiment_score,sentiment
0,Average quality for the price.,3,0.0000,Neutral
1,Excellent quality and works perfectly.,5,0.8360,Positive
2,Exactly as described. Very satisfied.,4,0.4754,Positive
3,Exactly as described. Very satisfied.,4,0.4754,Positive
4,Very happy with this purchase.,4,0.6115,Positive
5,Not worth the price. Very disappointed.,2,-0.6198,Negative
6,Excellent quality and works perfectly.,5,0.8360,Positive
7,The material feels cheap and uncomfortable.,1,-0.3818,Negative
8,Exactly as described. Very satisfied.,4,0.4754,Positive
9,Fast delivery and great product.,4,0.6249,Positive


In [10]:
sentiment_counts = reviews_df["sentiment"].value_counts()

print("Sentiment Distribution:")
print(sentiment_counts)

print("\nSentiment Percentage:")
print((sentiment_counts / len(reviews_df) * 100).round(2))

Sentiment Distribution:
sentiment
Positive    181
Negative     56
Neutral      13
Name: count, dtype: int64

Sentiment Percentage:
sentiment
Positive    72.4
Negative    22.4
Neutral      5.2
Name: count, dtype: float64


In [11]:
sentiment_rating = pd.crosstab(
    reviews_df["rating"],
    reviews_df["sentiment"]
)

sentiment_rating

sentiment,Negative,Neutral,Positive
rating,,,
1,21,0,0
2,21,0,0
3,14,13,34
4,0,0,73
5,0,0,74


In [12]:
mismatch_reviews = reviews_df[
    (
        (reviews_df["rating"] >= 4) &
        (reviews_df["sentiment"] == "Negative")
    ) |
    (
        (reviews_df["rating"] <= 2) &
        (reviews_df["sentiment"] == "Positive")
    )
]

print("Total mismatched reviews:", len(mismatch_reviews))

mismatch_reviews[
    ["review_id", "rating", "review_text", "sentiment"]
]

Total mismatched reviews: 0


,review_id,rating,review_text,sentiment


In [13]:
negative_reviews = reviews_df[
    reviews_df["sentiment"] == "Negative"
]

print("Total negative reviews:", len(negative_reviews))

negative_reviews[
    ["review_id", "product_id", "rating", "review_text"]
].head(15)

Total negative reviews: 56


,review_id,product_id,rating,review_text
5,R0006,P005,2,Not worth the price. Very disappointed.
7,R0008,P003,1,The material feels cheap and uncomfortable.
12,R0013,P015,1,Stopped working after two days.
22,R0023,P025,2,Not worth the price. Very disappointed.
25,R0026,P008,3,"Works as expected, nothing special."
27,R0028,P018,1,Stopped working after two days.
29,R0030,P012,1,Stopped working after two days.
30,R0031,P018,2,Not worth the price. Very disappointed.
35,R0036,P025,2,Not worth the price. Very disappointed.
38,R0039,P015,2,Not worth the price. Very disappointed.


In [14]:
def identify_complaint(text):
    text = text.lower()

    if any(word in text for word in ["price", "expensive", "cost"]):
        return "Pricing"

    elif any(word in text for word in ["cheap", "material", "uncomfortable"]):
        return "Material Quality"

    elif any(word in text for word in ["stopped working", "broken", "not working"]):
        return "Product Failure"

    elif any(word in text for word in ["poor quality", "bad quality"]):
        return "Poor Quality"

    elif any(word in text for word in ["nothing special", "average"]):
        return "Average Experience"

    else:
        return "Other"


negative_reviews = reviews_df[
    reviews_df["sentiment"] == "Negative"
].copy()

negative_reviews["complaint_category"] = (
    negative_reviews["review_text"].apply(identify_complaint)
)

negative_reviews[
    ["review_text", "complaint_category"]
].head(15)

,review_text,complaint_category
5,Not worth the price. Very disappointed.,Pricing
7,The material feels cheap and uncomfortable.,Material Quality
12,Stopped working after two days.,Product Failure
22,Not worth the price. Very disappointed.,Pricing
25,"Works as expected, nothing special.",Average Experience
27,Stopped working after two days.,Product Failure
29,Stopped working after two days.,Product Failure
30,Not worth the price. Very disappointed.,Pricing
35,Not worth the price. Very disappointed.,Pricing
38,Not worth the price. Very disappointed.,Pricing


In [15]:
complaint_counts = (
    negative_reviews["complaint_category"]
    .value_counts()
    .reset_index()
)

complaint_counts.columns = ["Complaint Category", "Review Count"]

complaint_counts["Percentage"] = (
    complaint_counts["Review Count"] / len(negative_reviews) * 100
).round(2)

complaint_counts

,Complaint Category,Review Count,Percentage
0,Average Experience,14,25.00
1,Other,13,23.21
2,Poor Quality,10,17.86
3,Pricing,9,16.07
4,Product Failure,6,10.71
5,Material Quality,4,7.14


In [16]:
other_reviews = negative_reviews[
    negative_reviews["complaint_category"] == "Other"
]

other_reviews[
    ["review_id", "rating", "review_text"]
]

,review_id,rating,review_text
73,R0074,2,Product arrived damaged.
92,R0093,2,Product arrived damaged.
103,R0104,2,Delivery was late and packaging was damaged.
110,R0111,1,Battery life is terrible.
126,R0127,1,Battery life is terrible.
184,R0185,2,Delivery was late and packaging was damaged.
197,R0198,1,Product arrived damaged.
199,R0200,2,Battery life is terrible.
206,R0207,1,Battery life is terrible.
209,R0210,2,Product arrived damaged.


In [17]:
def identify_complaint(text):
    text = text.lower()

    if any(word in text for word in ["price", "expensive", "cost"]):
        return "Pricing"

    elif any(word in text for word in ["cheap", "material", "uncomfortable"]):
        return "Material Quality"

    elif any(word in text for word in ["stopped working", "broken", "not working"]):
        return "Product Failure"

    elif any(word in text for word in ["poor quality", "bad quality"]):
        return "Poor Quality"

    elif any(word in text for word in ["nothing special", "average"]):
        return "Average Experience"

    elif any(word in text for word in ["damaged"]):
        if "delivery was late" in text or "packaging" in text:
            return "Delivery & Packaging"
        return "Damaged Product"

    elif any(word in text for word in ["battery"]):
        return "Battery Performance"

    else:
        return "Other"

In [18]:
negative_reviews["complaint_category"] = (
    negative_reviews["review_text"].apply(identify_complaint)
)

negative_reviews["complaint_category"].value_counts()

complaint_category
Average Experience      14
Poor Quality            10
Pricing                  9
Product Failure          6
Damaged Product          5
Battery Performance      5
Material Quality         4
Delivery & Packaging     3
Name: count, dtype: int64

In [19]:
negative_reviews[
    negative_reviews["complaint_category"] == "Average Experience"
][["review_id", "rating", "review_text", "sentiment_score"]]

,review_id,rating,review_text,sentiment_score
25,R0026,3,"Works as expected, nothing special.",-0.3089
59,R0060,3,"Works as expected, nothing special.",-0.3089
61,R0062,3,"Works as expected, nothing special.",-0.3089
66,R0067,3,"Works as expected, nothing special.",-0.3089
121,R0122,3,"Works as expected, nothing special.",-0.3089
148,R0149,3,"Works as expected, nothing special.",-0.3089
152,R0153,3,"Works as expected, nothing special.",-0.3089
168,R0169,3,"Works as expected, nothing special.",-0.3089
191,R0192,3,"Works as expected, nothing special.",-0.3089
196,R0197,3,"Works as expected, nothing special.",-0.3089


In [20]:
def corrected_sentiment(row):
    text = row["review_text"].lower().strip()

    if text == "works as expected, nothing special.":
        return "Neutral"

    score = row["sentiment_score"]

    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"


reviews_df["corrected_sentiment"] = reviews_df.apply(
    corrected_sentiment,
    axis=1
)

In [21]:
reviews_df["corrected_sentiment"].value_counts()

corrected_sentiment
Positive    181
Negative     42
Neutral      27
Name: count, dtype: int64

In [22]:
pd.crosstab(
    reviews_df["sentiment"],
    reviews_df["corrected_sentiment"]
)

corrected_sentiment,Negative,Neutral,Positive
sentiment,,,
Negative,42,14,0
Neutral,0,13,0
Positive,0,0,181


In [23]:
negative_reviews = reviews_df[
    reviews_df["corrected_sentiment"] == "Negative"
].copy()

print("Corrected negative reviews:", len(negative_reviews))

Corrected negative reviews: 42


In [24]:
negative_reviews["complaint_category"] = (
    negative_reviews["review_text"].apply(identify_complaint)
)

complaint_counts = (
    negative_reviews["complaint_category"]
    .value_counts()
    .reset_index()
)

complaint_counts.columns = ["Complaint Category", "Review Count"]

complaint_counts["Percentage"] = (
    complaint_counts["Review Count"] / len(negative_reviews) * 100
).round(2)

complaint_counts

,Complaint Category,Review Count,Percentage
0,Poor Quality,10,23.81
1,Pricing,9,21.43
2,Product Failure,6,14.29
3,Damaged Product,5,11.90
4,Battery Performance,5,11.90
5,Material Quality,4,9.52
6,Delivery & Packaging,3,7.14


In [25]:
# Fetch product details from MySQL
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password=db_password,
    database="ecommerce_intelligence"
)

products_df = pd.read_sql(
    "SELECT product_id, product_name, category, brand, unit_price FROM products",
    connection
)

connection.close()

# Merge negative reviews with product details
negative_product_analysis = negative_reviews.merge(
    products_df,
    on="product_id",
    how="left"
)

negative_product_analysis[
    ["product_name", "category", "rating",
     "review_text", "complaint_category"]
].head(10)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_3128\2661059072.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  products_df = pd.read_sql(


,product_name,category,rating,review_text,complaint_category
0,Power Bank,Electronics,2,Not worth the price. Very disappointed.,Pricing
1,Smart Watch,Electronics,1,The material feels cheap and uncomfortable.,Material Quality
2,Coffee Mug,Home,1,Stopped working after two days.,Product Failure
3,Shampoo,Beauty,2,Not worth the price. Very disappointed.,Pricing
4,Resistance Bands,Fitness,1,Stopped working after two days.,Product Failure
5,Travel Bottle,Home,1,Stopped working after two days.,Product Failure
6,Resistance Bands,Fitness,2,Not worth the price. Very disappointed.,Pricing
7,Shampoo,Beauty,2,Not worth the price. Very disappointed.,Pricing
8,Coffee Mug,Home,2,Not worth the price. Very disappointed.,Pricing
9,Face Moisturizer,Beauty,2,Poor quality and disappointing experience.,Poor Quality


In [26]:
product_complaints = (
    negative_product_analysis
    .groupby(["product_id", "product_name", "category"])
    .agg(
        total_complaints=("review_id", "count"),
        average_rating=("rating", "mean")
    )
    .reset_index()
    .sort_values(
        by=["total_complaints", "average_rating"],
        ascending=[False, True]
    )
)

product_complaints["average_rating"] = (
    product_complaints["average_rating"].round(2)
)

product_complaints

,product_id,product_name,category,total_complaints,average_rating
2,P005,Power Bank,Electronics,4,1.75
8,P015,Coffee Mug,Home,4,1.75
20,P030,Electric Kettle,Appliances,3,1.00
1,P003,Smart Watch,Electronics,3,1.33
11,P018,Resistance Bands,Fitness,3,1.33
12,P019,Cotton T-Shirt,Fashion,3,1.67
17,P025,Shampoo,Beauty,3,1.67
4,P010,Screen Protector,Accessories,2,1.00
3,P008,Wireless Mouse,Electronics,2,1.50
9,P016,Running Shoes,Fashion,2,1.50


In [27]:
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password=db_password,
    database="ecommerce_intelligence"
)

orders_df = pd.read_sql(
    """
    SELECT
        order_id,
        product_id,
        order_date,
        quantity,
        unit_price,
        order_status
    FROM orders
    """,
    connection
)

connection.close()

print("Total orders fetched:", len(orders_df))

orders_df.head()

Total orders fetched: 500


C:\Users\Administrator\AppData\Local\Temp\ipykernel_3128\2399767614.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  orders_df = pd.read_sql(


,order_id,product_id,order_date,quantity,unit_price,order_status
0,O0001,P021,2025-02-27,1,2499.0,Delivered
1,O0002,P008,2025-04-25,2,699.0,Delivered
2,O0003,P022,2025-10-07,1,699.0,Returned
3,O0004,P014,2025-01-17,1,349.0,Delivered
4,O0005,P007,2025-04-30,1,2999.0,Returned


In [28]:
orders_df["order_status"].value_counts()

order_status
Delivered    291
Cancelled    114
Returned      95
Name: count, dtype: int64

In [29]:
orders_df["order_status"].value_counts(normalize=True).mul(100).round(2)

order_status
Delivered    58.2
Cancelled    22.8
Returned     19.0
Name: proportion, dtype: float64

In [30]:
delivered_orders = orders_df[
    orders_df["order_status"] == "Delivered"
].copy()

delivered_orders["revenue"] = (
    delivered_orders["quantity"] *
    delivered_orders["unit_price"]
)

product_sales = (
    delivered_orders
    .groupby("product_id")
    .agg(
        total_quantity_sold=("quantity", "sum"),
        total_orders=("order_id", "count"),
        total_revenue=("revenue", "sum")
    )
    .reset_index()
)

product_sales = product_sales.sort_values(
    by="total_revenue",
    ascending=False
)

product_sales.head(10)

,product_id,total_quantity_sold,total_orders,total_revenue
26,P027,30,12,164970.0
28,P029,24,11,95976.0
2,P003,30,12,74970.0
6,P007,22,8,65978.0
1,P002,32,12,63968.0
20,P021,19,11,47481.0
10,P011,30,13,38970.0
15,P016,16,8,35184.0
4,P005,32,13,31968.0
19,P020,20,10,29980.0


In [31]:
# Merge product sales with product details
product_sales_analysis = product_sales.merge(
    products_df,
    on="product_id",
    how="left"
)

# Merge with product complaint analysis
final_product_analysis = product_sales_analysis.merge(
    product_complaints[
        ["product_id", "total_complaints", "average_rating"]
    ],
    on="product_id",
    how="left"
)

# Products without negative reviews get zero complaints
final_product_analysis["total_complaints"] = (
    final_product_analysis["total_complaints"].fillna(0).astype(int)
)

final_product_analysis["average_rating"] = (
    final_product_analysis["average_rating"].fillna(0)
)

final_product_analysis = final_product_analysis.sort_values(
    by="total_revenue",
    ascending=False
)

final_product_analysis[
    [
        "product_name",
        "category",
        "total_quantity_sold",
        "total_revenue",
        "total_complaints",
        "average_rating"
    ]
].head(15)

,product_name,category,total_quantity_sold,total_revenue,total_complaints,average_rating
0,Office Chair,Furniture,30,164970.0,1,1.00
1,Air Fryer,Appliances,24,95976.0,0,0.00
2,Smart Watch,Electronics,30,74970.0,3,1.33
3,Mechanical Keyboard,Electronics,22,65978.0,0,0.00
4,Bluetooth Speaker,Electronics,32,63968.0,0,0.00
5,Cookware Set,Home,19,47481.0,2,2.00
6,Backpack,Fashion,30,38970.0,1,2.00
7,Running Shoes,Fashion,16,35184.0,2,1.50
8,Power Bank,Electronics,32,31968.0,4,1.75
9,Hoodie,Fashion,20,29980.0,0,0.00


In [32]:
# Total reviews and average rating for every product
product_review_summary = (
    reviews_df
    .groupby("product_id")
    .agg(
        total_reviews=("review_id", "count"),
        overall_average_rating=("rating", "mean")
    )
    .reset_index()
)

# Merge review summary into sales and complaint analysis
final_product_analysis = final_product_analysis.merge(
    product_review_summary,
    on="product_id",
    how="left"
)

# Calculate negative review percentage
final_product_analysis["complaint_rate"] = (
    final_product_analysis["total_complaints"]
    / final_product_analysis["total_reviews"] * 100
).round(2)

final_product_analysis["overall_average_rating"] = (
    final_product_analysis["overall_average_rating"].round(2)
)

# View the combined analysis
final_product_analysis[
    [
        "product_name",
        "total_quantity_sold",
        "total_revenue",
        "total_reviews",
        "total_complaints",
        "complaint_rate",
        "overall_average_rating"
    ]
].sort_values(
    by="complaint_rate",
    ascending=False
).head(15)

,product_name,total_quantity_sold,total_revenue,total_reviews,total_complaints,complaint_rate,overall_average_rating
23,Coffee Mug,32,7968.0,13,4,30.77,3.08
20,Resistance Bands,21,10479.0,10,3,30.00,2.90
18,Cotton T-Shirt,19,11381.0,10,3,30.00,3.60
26,Shampoo,19,5681.0,10,3,30.00,3.10
8,Power Bank,32,31968.0,14,4,28.57,3.64
22,Face Moisturizer,22,8778.0,7,2,28.57,2.86
2,Smart Watch,30,74970.0,13,3,23.08,3.31
5,Cookware Set,19,47481.0,9,2,22.22,3.78
27,Screen Protector,25,4975.0,10,2,20.00,3.40
7,Running Shoes,16,35184.0,10,2,20.00,3.10


In [33]:
median_revenue = final_product_analysis["total_revenue"].median()

def classify_product(row):
    if (
        row["total_revenue"] >= median_revenue
        and row["complaint_rate"] > 20
        and row["total_reviews"] >= 5
    ):
        return "High Sales - High Complaints"

    elif row["complaint_rate"] > 20:
        return "High Complaint Rate"

    elif row["total_revenue"] >= median_revenue:
        return "Strong Sales"

    else:
        return "Monitor"

final_product_analysis["business_priority"] = (
    final_product_analysis.apply(classify_product, axis=1)
)

final_product_analysis[
    [
        "product_name",
        "total_revenue",
        "total_reviews",
        "total_complaints",
        "complaint_rate",
        "business_priority"
    ]
].sort_values(
    by=["business_priority", "total_revenue"],
    ascending=[True, False]
)

,product_name,total_revenue,total_reviews,total_complaints,complaint_rate,business_priority
18,Cotton T-Shirt,11381.0,10,3,30.00,High Complaint Rate
20,Resistance Bands,10479.0,10,3,30.00,High Complaint Rate
22,Face Moisturizer,8778.0,7,2,28.57,High Complaint Rate
23,Coffee Mug,7968.0,13,4,30.77,High Complaint Rate
26,Shampoo,5681.0,10,3,30.00,High Complaint Rate
2,Smart Watch,74970.0,13,3,23.08,High Sales - High Complaints
5,Cookware Set,47481.0,9,2,22.22,High Sales - High Complaints
8,Power Bank,31968.0,14,4,28.57,High Sales - High Complaints
15,Travel Bottle,17465.0,6,1,16.67,Monitor
16,Desk Lamp,17081.0,10,1,10.00,Monitor


In [34]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Sentence Transformer model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sentence Transformer model loaded successfully!


In [35]:
# Extract review text
review_texts = reviews_df["review_text"].fillna("").tolist()

# Generate sentence embeddings
review_embeddings = model.encode(
    review_texts,
    show_progress_bar=True
)

print("Embedding generation completed!")
print("Number of reviews:", len(review_texts))
print("Embedding shape:", review_embeddings.shape)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Embedding generation completed!
Number of reviews: 250
Embedding shape: (250, 384)


In [36]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate similarity between all reviews
similarity_matrix = cosine_similarity(review_embeddings)

print("Similarity matrix shape:", similarity_matrix.shape)
print("Similarity calculation completed!")

Similarity matrix shape: (250, 250)
Similarity calculation completed!


In [37]:
import numpy as np

# Get the upper triangle to avoid duplicate pairs
similarity_pairs = []

for i in range(len(review_texts)):
    for j in range(i + 1, len(review_texts)):
        similarity_pairs.append({
            "review_1": review_texts[i],
            "review_2": review_texts[j],
            "similarity": similarity_matrix[i, j]
        })

# Convert into DataFrame
similarity_df = pd.DataFrame(similarity_pairs)

# Show the 15 most similar review pairs
similarity_df = similarity_df.sort_values(
    by="similarity",
    ascending=False
)

similarity_df.head(15)


,review_1,review_2,similarity
20539,Delivery was late and packaging was damaged.,Delivery was late and packaging was damaged.,1.0
29044,Delivery was late and packaging was damaged.,Delivery was late and packaging was damaged.,1.0
20615,Poor quality and disappointing experience.,Poor quality and disappointing experience.,1.0
28164,Decent product but could be improved.,Decent product but could be improved.,1.0
6276,Decent product but could be improved.,Decent product but could be improved.,1.0
20877,Poor quality and disappointing experience.,Poor quality and disappointing experience.,1.0
14413,"Works as expected, nothing special.","Works as expected, nothing special.",1.0
23039,Excellent quality and works perfectly.,Excellent quality and works perfectly.,1.0
28201,Fast delivery and great product.,Fast delivery and great product.,1.0
31025,Excellent quality and works perfectly.,Excellent quality and works perfectly.,1.0


In [38]:
# Keep only pairs with different review text
different_review_pairs = similarity_df[
    similarity_df["review_1"] != similarity_df["review_2"]
].copy()

# Sort by similarity
different_review_pairs = different_review_pairs.sort_values(
    by="similarity",
    ascending=False
)

# Display top 15 different review pairs
different_review_pairs.head(15)

,review_1,review_2,similarity
29020,Delivery was late and packaging was damaged.,Product arrived damaged.,0.69557
18813,Product arrived damaged.,Delivery was late and packaging was damaged.,0.69557
20515,Delivery was late and packaging was damaged.,Product arrived damaged.,0.69557
18878,Product arrived damaged.,Delivery was late and packaging was damaged.,0.69557
30848,Product arrived damaged.,Delivery was late and packaging was damaged.,0.69557
30344,Product arrived damaged.,Delivery was late and packaging was damaged.,0.69557
28992,Delivery was late and packaging was damaged.,Product arrived damaged.,0.69557
20487,Delivery was late and packaging was damaged.,Product arrived damaged.,0.69557
29004,Delivery was late and packaging was damaged.,Product arrived damaged.,0.69557
15659,Product arrived damaged.,Delivery was late and packaging was damaged.,0.69557


In [39]:
# Remove duplicate text combinations
unique_pairs = different_review_pairs.drop_duplicates(
    subset=["review_1", "review_2"]
).copy()

# Show the top 20 unique pairs
unique_pairs.head(20).reset_index(drop=True)

,review_1,review_2,similarity
0,Delivery was late and packaging was damaged.,Product arrived damaged.,0.695570
1,Product arrived damaged.,Delivery was late and packaging was damaged.,0.695570
2,Decent product but could be improved.,Fast delivery and great product.,0.494206
3,Fast delivery and great product.,Decent product but could be improved.,0.494206
4,Average quality for the price.,Excellent quality and works perfectly.,0.492245
5,Excellent quality and works perfectly.,Average quality for the price.,0.492245
6,Excellent quality and works perfectly.,Very happy with this purchase.,0.491467
7,Very happy with this purchase.,Excellent quality and works perfectly.,0.491467
8,Very happy with this purchase.,Not worth the price. Very disappointed.,0.486831
9,Not worth the price. Very disappointed.,Very happy with this purchase.,0.486831


In [40]:
# Remove reversed duplicate pairs
unique_pairs = different_review_pairs.copy()

unique_pairs["pair_key"] = unique_pairs.apply(
    lambda row: tuple(sorted([
        row["review_1"],
        row["review_2"]
    ])),
    axis=1
)

unique_pairs = unique_pairs.drop_duplicates(
    subset="pair_key"
).drop(columns="pair_key")

unique_pairs = unique_pairs.sort_values(
    by="similarity",
    ascending=False
)

unique_pairs.head(15).reset_index(drop=True)

,review_1,review_2,similarity
0,Delivery was late and packaging was damaged.,Product arrived damaged.,0.695570
1,Decent product but could be improved.,Fast delivery and great product.,0.494206
2,Average quality for the price.,Excellent quality and works perfectly.,0.492245
3,Excellent quality and works perfectly.,Very happy with this purchase.,0.491467
4,Very happy with this purchase.,Not worth the price. Very disappointed.,0.486831
5,Average quality for the price.,Good value for money. Would recommend.,0.485689
6,Fast delivery and great product.,Very happy with this purchase.,0.482211
7,Good value for money. Would recommend.,Not worth the price. Very disappointed.,0.477868
8,Not worth the price. Very disappointed.,Poor quality and disappointing experience.,0.473912
9,Excellent quality and works perfectly.,Decent product but could be improved.,0.469354


In [41]:
from sklearn.cluster import KMeans

# Create 8 clusters of similar reviews
kmeans = KMeans(
    n_clusters=8,
    random_state=42,
    n_init=10
)

# Assign cluster labels
reviews_df["cluster"] = kmeans.fit_predict(review_embeddings)

print("Clustering completed!")
print(reviews_df["cluster"].value_counts().sort_index())

Clustering completed!
cluster
0    31
1    31
2    59
3    29
4    23
5    30
6    27
7    20
Name: count, dtype: int64


In [42]:
for cluster_num in sorted(reviews_df["cluster"].unique()):

    print(f"\n{'='*60}")
    print(f"CLUSTER {cluster_num}")
    print(f"{'='*60}")

    cluster_reviews = reviews_df[
        reviews_df["cluster"] == cluster_num
    ]

    for review in cluster_reviews["review_text"].head(10):
        print("-", review)


CLUSTER 0
- Good value for money. Would recommend.
- Good value for money. Would recommend.
- Good value for money. Would recommend.
- Good value for money. Would recommend.
- Good value for money. Would recommend.
- Good value for money. Would recommend.
- Good value for money. Would recommend.
- Good value for money. Would recommend.
- Good value for money. Would recommend.
- Good value for money. Would recommend.

CLUSTER 1
- Exactly as described. Very satisfied.
- Exactly as described. Very satisfied.
- Exactly as described. Very satisfied.
- Exactly as described. Very satisfied.
- Exactly as described. Very satisfied.
- Exactly as described. Very satisfied.
- Exactly as described. Very satisfied.
- Exactly as described. Very satisfied.
- Exactly as described. Very satisfied.
- Exactly as described. Very satisfied.

CLUSTER 2
- Fast delivery and great product.
- Fast delivery and great product.
- Fast delivery and great product.
- Decent product but could be improved.
- Decent pro

In [43]:
cluster_summary = (
    reviews_df
    .groupby(["cluster", "review_text"])
    .size()
    .reset_index(name="review_count")
    .sort_values(
        ["cluster", "review_count"],
        ascending=[True, False]
    )
)

for cluster_num in sorted(reviews_df["cluster"].unique()):

    print(f"\n--- CLUSTER {cluster_num} ---")

    print(
        cluster_summary[
            cluster_summary["cluster"] == cluster_num
        ][["review_text", "review_count"]].to_string(index=False)
    )


--- CLUSTER 0 ---
                           review_text  review_count
Good value for money. Would recommend.            31

--- CLUSTER 1 ---
                          review_text  review_count
Exactly as described. Very satisfied.            31

--- CLUSTER 2 ---
                                 review_text  review_count
            Fast delivery and great product.            35
       Decent product but could be improved.            16
                    Product arrived damaged.             5
Delivery was late and packaging was damaged.             3

--- CLUSTER 3 ---
                           review_text  review_count
Excellent quality and works perfectly.            29

--- CLUSTER 4 ---
                          review_text  review_count
The product is okay for everyday use.            18
            Battery life is terrible.             5

--- CLUSTER 5 ---
                            review_text  review_count
         Very happy with this purchase.            21
Not worth t

In [44]:
cluster_sentiment = pd.crosstab(
    reviews_df["cluster"],
    reviews_df["corrected_sentiment"]
)

cluster_sentiment["Total"] = cluster_sentiment.sum(axis=1)

cluster_sentiment

corrected_sentiment,Negative,Neutral,Positive,Total
cluster,,,,
0,0,0,31,31
1,0,0,31,31
2,8,0,51,59
3,0,0,29,29
4,5,0,18,23
5,9,0,21,30
6,14,13,0,27
7,6,14,0,20


In [45]:
cluster_analysis = (
    reviews_df
    .groupby("cluster")
    .agg(
        total_reviews=("review_id", "count"),
        negative_reviews=(
            "corrected_sentiment",
            lambda x: (x == "Negative").sum()
        ),
        neutral_reviews=(
            "corrected_sentiment",
            lambda x: (x == "Neutral").sum()
        ),
        positive_reviews=(
            "corrected_sentiment",
            lambda x: (x == "Positive").sum()
        )
    )
    .reset_index()
)

cluster_analysis["negative_percentage"] = (
    cluster_analysis["negative_reviews"]
    / cluster_analysis["total_reviews"] * 100
).round(2)

cluster_analysis.sort_values(
    by="negative_percentage",
    ascending=False
)

,cluster,total_reviews,negative_reviews,neutral_reviews,positive_reviews,negative_percentage
6,6,27,14,13,0,51.85
5,5,30,9,0,21,30.00
7,7,20,6,14,0,30.00
4,4,23,5,0,18,21.74
2,2,59,8,0,51,13.56
3,3,29,0,0,29,0.00
0,0,31,0,0,31,0.00
1,1,31,0,0,31,0.00


In [46]:
for cluster_id in [2, 4, 5, 6, 7]:

    print(f"\n========== CLUSTER {cluster_id} ==========")

    cluster_reviews = reviews_df[
        reviews_df["cluster"] == cluster_id
    ]

    for sentiment in ["Negative", "Neutral", "Positive"]:

        selected_reviews = cluster_reviews[
            cluster_reviews["corrected_sentiment"] == sentiment
        ]

        if len(selected_reviews) > 0:
            print(f"\n{sentiment} REVIEWS:")

            for review in selected_reviews["review_text"].unique():
                print("-", review)


========== CLUSTER 2 ==========

Negative REVIEWS:
- Product arrived damaged.
- Delivery was late and packaging was damaged.

Positive REVIEWS:
- Fast delivery and great product.
- Decent product but could be improved.

========== CLUSTER 4 ==========

Negative REVIEWS:
- Battery life is terrible.

Positive REVIEWS:
- The product is okay for everyday use.

========== CLUSTER 5 ==========

Negative REVIEWS:
- Not worth the price. Very disappointed.

Positive REVIEWS:
- Very happy with this purchase.

========== CLUSTER 6 ==========

Negative REVIEWS:
- The material feels cheap and uncomfortable.
- Poor quality and disappointing experience.

Neutral REVIEWS:
- Average quality for the price.

========== CLUSTER 7 ==========

Negative REVIEWS:
- Stopped working after two days.

Neutral REVIEWS:
- Works as expected, nothing special.


In [47]:
cluster_labels = {
    0: "Value for Money",
    1: "Customer Satisfaction",
    2: "Delivery & Product Condition",
    3: "Product Quality & Performance",
    4: "Battery & Everyday Usage",
    5: "Pricing & Purchase Satisfaction",
    6: "Material & Quality Concerns",
    7: "Product Reliability"
}

reviews_df["cluster_theme"] = reviews_df["cluster"].map(cluster_labels)

reviews_df[
    ["review_id", "review_text", "corrected_sentiment",
     "cluster", "cluster_theme"]
].head(10)

,review_id,review_text,corrected_sentiment,cluster,cluster_theme
0,R0001,Average quality for the price.,Neutral,6,Material & Quality Concerns
1,R0002,Excellent quality and works perfectly.,Positive,3,Product Quality & Performance
2,R0003,Exactly as described. Very satisfied.,Positive,1,Customer Satisfaction
3,R0004,Exactly as described. Very satisfied.,Positive,1,Customer Satisfaction
4,R0005,Very happy with this purchase.,Positive,5,Pricing & Purchase Satisfaction
5,R0006,Not worth the price. Very disappointed.,Negative,5,Pricing & Purchase Satisfaction
6,R0007,Excellent quality and works perfectly.,Positive,3,Product Quality & Performance
7,R0008,The material feels cheap and uncomfortable.,Negative,6,Material & Quality Concerns
8,R0009,Exactly as described. Very satisfied.,Positive,1,Customer Satisfaction
9,R0010,Fast delivery and great product.,Positive,2,Delivery & Product Condition


In [48]:
reviews_df["cluster_theme"].value_counts()

cluster_theme
Delivery & Product Condition       59
Customer Satisfaction              31
Value for Money                    31
Pricing & Purchase Satisfaction    30
Product Quality & Performance      29
Material & Quality Concerns        27
Battery & Everyday Usage           23
Product Reliability                20
Name: count, dtype: int64

In [49]:
reviews_df.to_csv(
    "../data/clean/reviews_with_sentiment_clusters.csv",
    index=False
)

print("Review analysis dataset saved successfully!")

Review analysis dataset saved successfully!


In [50]:
reviews_df["complaint_category"] = reviews_df.apply(
    lambda row: identify_complaint(row["review_text"])
    if row["corrected_sentiment"] == "Negative"
    else None,
    axis=1
)

In [51]:
reviews_df[
    reviews_df["corrected_sentiment"] == "Negative"
][[
    "review_id",
    "review_text",
    "complaint_category"
]].head(10)

,review_id,review_text,complaint_category
5,R0006,Not worth the price. Very disappointed.,Pricing
7,R0008,The material feels cheap and uncomfortable.,Material Quality
12,R0013,Stopped working after two days.,Product Failure
22,R0023,Not worth the price. Very disappointed.,Pricing
27,R0028,Stopped working after two days.,Product Failure
29,R0030,Stopped working after two days.,Product Failure
30,R0031,Not worth the price. Very disappointed.,Pricing
35,R0036,Not worth the price. Very disappointed.,Pricing
38,R0039,Not worth the price. Very disappointed.,Pricing
53,R0054,Poor quality and disappointing experience.,Poor Quality


In [52]:
negative_reviews_export = reviews_df[
    reviews_df["corrected_sentiment"] == "Negative"
][[
    "review_id",
    "review_text",
    "complaint_category"
]].copy()

print("Total negative reviews:", len(negative_reviews_export))

Total negative reviews: 42


In [53]:
negative_reviews_export.to_csv(
    "../data/clean/negative_reviews_for_genai.csv",
    index=False
)

In [54]:
print("Total exported reviews:", len(negative_reviews_export))

print("\nComplaint category distribution:")
print(negative_reviews_export["complaint_category"].value_counts())

print("\nMissing values:")
print(negative_reviews_export.isnull().sum())

Total exported reviews: 42

Complaint category distribution:
complaint_category
Poor Quality            10
Pricing                  9
Product Failure          6
Damaged Product          5
Battery Performance      5
Material Quality         4
Delivery & Packaging     3
Name: count, dtype: int64

Missing values:
review_id             0
review_text           0
complaint_category    0
dtype: int64


In [55]:
negative_reviews_export.to_csv(
    "../data/clean/negative_reviews_for_genai.csv",
    index=False
)

In [56]:
import os

file_path = "../data/clean/negative_reviews_for_genai.csv"

print("Current notebook folder:", os.getcwd())
print("File exists:", os.path.exists(file_path))
print("Full file path:", os.path.abspath(file_path))

Current notebook folder: c:\Users\Administrator\Documents\ecommerce_ai_project\notebooks
File exists: True
Full file path: c:\Users\Administrator\Documents\ecommerce_ai_project\data\clean\negative_reviews_for_genai.csv


In [57]:
import os

clean_folder = "../data/clean"

for file in os.listdir(clean_folder):
    print(file)

negative_reviews_for_genai.csv
orders_clean.csv
products_clean.csv
prompt_engineering_comparison.csv
reviews_clean.csv
reviews_with_sentiment_clusters.csv


In [58]:
print("PRODUCTS:")
print(products_df.columns.tolist())

print("\nORDERS:")
print(orders_df.columns.tolist())

print("\nREVIEWS:")
print(reviews_df.columns.tolist())

print("\nFINAL PRODUCT ANALYSIS:")
print(final_product_analysis.columns.tolist())

PRODUCTS:
['product_id', 'product_name', 'category', 'brand', 'unit_price']

ORDERS:
['order_id', 'product_id', 'order_date', 'quantity', 'unit_price', 'order_status']

REVIEWS:
['review_id', 'product_id', 'review_date', 'rating', 'review_text', 'sentiment_score', 'sentiment', 'corrected_sentiment', 'cluster', 'cluster_theme', 'complaint_category']

FINAL PRODUCT ANALYSIS:
['product_id', 'total_quantity_sold', 'total_orders', 'total_revenue', 'product_name', 'category', 'brand', 'unit_price', 'total_complaints', 'average_rating', 'total_reviews', 'overall_average_rating', 'complaint_rate', 'business_priority']


In [59]:
import os

# Create output folder
os.makedirs("../data/clean", exist_ok=True)

# 1. Final product master
products_df.to_csv(
    "../data/clean/products_final.csv",
    index=False
)

# 2. Final sales analysis
product_sales_analysis.to_csv(
    "../data/clean/sales_analysis_final.csv",
    index=False
)

# 3. Final review analysis
reviews_df.to_csv(
    "../data/clean/reviews_analysis_final.csv",
    index=False
)

# 4. Final product-level analysis
final_product_analysis.to_csv(
    "../data/clean/product_analysis_final.csv",
    index=False
)

print("All four final analytical datasets exported successfully!")

All four final analytical datasets exported successfully!


In [60]:
final_files = [
    "products_final.csv",
    "sales_analysis_final.csv",
    "reviews_analysis_final.csv",
    "product_analysis_final.csv"
]

for file in final_files:
    df = pd.read_csv(f"../data/clean/{file}")

    print(
        f"{file} | Rows: {len(df)} | Columns: {len(df.columns)}"
    )

products_final.csv | Rows: 30 | Columns: 5
sales_analysis_final.csv | Rows: 30 | Columns: 8
reviews_analysis_final.csv | Rows: 250 | Columns: 11
product_analysis_final.csv | Rows: 30 | Columns: 14


In [61]:
product_summary = final_product_analysis[
    [
        "product_id",
        "total_quantity_sold",
        "total_orders",
        "total_revenue",
        "total_complaints",
        "average_rating",
        "total_reviews",
        "overall_average_rating",
        "complaint_rate",
        "business_priority"
    ]
].copy()

product_summary.to_csv(
    "../data/clean/product_summary_final.csv",
    index=False
)

print("Product summary exported!")
print(product_summary.shape)
print(product_summary.head())

Product summary exported!
(30, 10)
  product_id  total_quantity_sold  total_orders  total_revenue  \
0       P027                   30            12       164970.0   
1       P029                   24            11        95976.0   
2       P003                   30            12        74970.0   
3       P007                   22             8        65978.0   
4       P002                   32            12        63968.0   

   total_complaints  average_rating  total_reviews  overall_average_rating  \
0                 1            1.00              7                    3.71   
1                 0            0.00              4                    4.00   
2                 3            1.33             13                    3.31   
3                 0            0.00              6                    4.00   
4                 0            0.00              4                    3.75   

   complaint_rate             business_priority  
0           14.29                  Strong Sales  

In [62]:
print("Product ID duplicates:",
      product_summary["product_id"].duplicated().sum())

print("Missing values:")
print(product_summary.isnull().sum())

print("\nBusiness priority distribution:")
print(product_summary["business_priority"].value_counts())

print("\nReview sentiment distribution:")
print(reviews_df["corrected_sentiment"].value_counts())

Product ID duplicates: 0
Missing values:
product_id                0
total_quantity_sold       0
total_orders              0
total_revenue             0
total_complaints          0
average_rating            0
total_reviews             0
overall_average_rating    0
complaint_rate            0
business_priority         0
dtype: int64

Business priority distribution:
business_priority
Strong Sales                    12
Monitor                         10
High Complaint Rate              5
High Sales - High Complaints     3
Name: count, dtype: int64

Review sentiment distribution:
corrected_sentiment
Positive    181
Negative     42
Neutral      27
Name: count, dtype: int64
